In [1]:
import os
import cv2
import pyewts
import numpy as np
from glob import glob
from tqdm import tqdm
from natsort import natsorted
from evaluate import load
from Utils import read_label, get_file_name, pad_image2, binarize_line, process_wylie_label
from Modules import OnnxInference

d:\Github\OCR-Inference\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def prepare_ocr_image2(
    line_image: np.array, target_width: int = 2000, target_height: int = 80
    ):
        if len(line_image.shape) > 2:
            line_image = cv2.cvtColor(line_image, cv2.COLOR_BGR2GRAY)

        image = pad_image2(
            line_image,
            target_width=target_width,
            target_height=target_height,
            padding="black",
        )
        
        image = image.reshape((1, target_height, target_width))

        #image = (image / 127.5) - 1.0
        image = image / 255.0
        image = image.astype(np.float32)
        return image

In [3]:
converter = pyewts.pyewts()
cer_scorer = load("cer")

In [22]:
#dataset_root = "Models/OCRModels"
#dataset = "DergeTenjur"
#chckpt = "2024_1_25_6_48"


"""dataset_root = "Models/OCRModels"
dataset = "LhasaKanjur"
chckpt = "Prodigy_2023_9_13_7_59"

model_config = f"{dataset_root}/{dataset}/{chckpt}/model_config.json"
ocr_inference = OnnxInference(config_file=model_config)
"""

"""
dataset_root = "Models/OCRModels"
dataset = "LhasaKanjur"
chckpt = "Prodigy_2023_9_13_7_59/EASTER2"
"""

dataset_root = "Models/OCRModels"
dataset = "LithangKanjur"
chckpt = "2024_1_20_8_23"

model_config = f"{dataset_root}/{dataset}/{chckpt}/model_config.json"
#ocr_model_config = f"{models_path}/{dataset}/model_config.json"

#ocr_model_config = "Models/OCRModels/DergeTenjur/November2023_v2/model_config.json"
ocr_inference = OnnxInference(config_file=model_config, mode="cuda")

In [5]:
test_set_index = f"{dataset_root}/{dataset}/Prodigy_2023_9_13_7_59/test_imgs.txt"

with open(test_set_index, "r") as f:
    entries = f.readlines()
    entries = [get_file_name(x) for x in entries]

print(entries[:10])
print(len(entries))

['18-1-289b_line_9874_1', '16-1-506b_line_9874_4', '12-1-374b_line_9874_6', '78-1-17a_line_9874_1', '11-1-135a_line_9874_5', '13-1-261a_line_9874_2', '13-1-320b_line_9874_2', '12-1-35b_line_9874_4', '14-1-188b_line_9874_3', '21-1-436b_line_9874_1']
16064


In [21]:
#dataset_path = "Data/W23703/Dataset"
#dataset_path = "D:/Datasets/Tibetan/LhasaKanjur/LhasaKanjurProdigy_v1_reviewed"
dataset_path = "D:/Datasets/Tibetan/OCR/LithangKanjur/DatasetFiltered"

all_images = natsorted(glob(f"{dataset_path}/lines/*.jpg"))
all_labels = natsorted(glob(f"{dataset_path}/transcriptions/*.txt"))

print(f"All Images: {len(all_images)}, All Labels: {len(all_images)}")

test_images = [f"{dataset_path}/lines/{x}.jpg" for x in entries]
test_labels = [f"{dataset_path}/transcriptions/{x}.txt" for x in entries]

print(f"Test Images: {len(test_images)}, Test Labels: {len(test_labels)}")

All Images: 520000, All Labels: 520000
Test Images: 16064, Test Labels: 16064


In [23]:
cer_scores = []

for image, label in tqdm(zip(test_images, test_labels), total=len(test_images)):
    test_img = cv2.imread(image)
    test_lbl = read_label(label)
    test_lbl = converter.toWylie(test_lbl)
    test_lbl = process_wylie_label(test_lbl)

    test_img = prepare_ocr_image2(test_img, target_height=ocr_inference._input_height)
    np_img_batch = np.array(test_img)
    np_img_batch = np.expand_dims(np_img_batch, axis=1)

    predicted_text, raw = ocr_inference.run(np_img_batch, replace_tsheg=" ")
    predicted_text = predicted_text[0].replace("/ ", "")
    predicted_text = predicted_text.replace(" ", "§")

    cer_score = cer_scorer.compute(predictions=[predicted_text], references=[test_lbl])
    cer_scores.append(cer_score)


mean_cer = np.mean(cer_scores)

print(f"Mean CER: {mean_cer}")
print(f"Max CER: {max(cer_scores)}")
print(f"Min CER: {min(cer_scores)}")

with open(f"{dataset_root}/{dataset}/{chckpt}/test_set_cer_scores.txt", "w") as f:
    for idx, c_score in enumerate(cer_scores):
            f.write(f"{c_score}, {test_labels[idx]}\n")

  0%|          | 0/16064 [00:00<?, ?it/s]


FileNotFoundError: [Errno 2] No such file or directory: 'D:/Datasets/Tibetan/OCR/LithangKanjur/DatasetFiltered/transcriptions/18-1-289b_line_9874_1.txt'

In [19]:
# single line image:
idx = 1234

test_img = cv2.imread(test_images[idx])
test_lbl = read_label(test_labels[idx])
test_lbl = converter.toWylie(test_lbl)
test_lbl = process_wylie_label(test_lbl)


test_img = prepare_ocr_image2(test_img, target_height=ocr_inference._input_height)
img_batch = np.array(test_img)
img_batch = np.expand_dims(img_batch, axis=1)

predicted_text, raw = ocr_inference.run(img_batch, replace_tsheg=" ")
predicted_text = predicted_text[0].replace("/ ", "")
predicted_text = predicted_text.replace(" ", "§")
print(test_lbl)
print(predicted_text)

cer_score = cer_scorer.compute(predictions=[predicted_text], references=[test_lbl])
print(cer_score)



'bul§lo//bzhin§bzang§/a§rog§kya/de§dag§khyod§kyir§gyur§cig/sbyin§pa§'dis§khyod§sems§kyi§rgyan§dang§/sems§kyi§yo§byad§dang§/rnal§'byor§gyi§tshogs§dang§/don§gyi§mchog§thob§pa§dang§/
'bul§lo/bzhin§bzang§a§rog§kyade§dag§khyod§kyir§gyur§cig§/sbyin§pa§'dis§khyod§sems§kyi§rgyan§dang§sems§kyi§yo§byad§dang§rnal§'byor§gyi§tshogs§dang§don§gyi§mchog§thob§pa§dang§/
0.03910614525139665


In [10]:
cer_score = cer_scorer.compute(predictions=predicted_text, references=[test_lbl])
print(cer_score)

1.1481481481481481


In [11]:
test_lbl

'འབུལ་ལོ།§།བཞིན་བཟང༌།§ཨ་རོག་ཀྱ།§དེ་དག་ཁྱོད་ཀྱིར་གྱུར་ཅིག§།སྦྱིན་པ་འདིས་ཁྱོད་སེམས་ཀྱི་རྒྱན་དང༌།§སེམས་ཀྱི་ཡོ་བྱད་དང༌།§རྣལ་འབྱོར་གྱི་ཚོགས་དང༌།§དོན་གྱི་མཆོག་ཐོབ་པ་དང༌།'

In [12]:
predicted_text

["'bul lo/ /bzhin bzang / a rog kya/ de dag khyod kyir gyur cig /sbyin pa 'dis khyod sems kyi rgyan dang / sems kyi yo byad dang / rnal 'byor gyi tshogs dang / don gyi mchog thob pa dang /"]

In [18]:
test_lbl = converter.toWylie(test_lbl)

In [19]:
test_lbl

"'bul lo/_/bzhin bzang*/_a rog kya/_de dag khyod kyir gyur cig_/sbyin pa 'dis khyod sems kyi rgyan dang*/_sems kyi yo byad dang*/_rnal 'byor gyi tshogs dang*/_don gyi mchog thob pa dang*/"